### Build Nationality Reference Table (`ref_nationalaty_regions`)

This notebook creates a **reference/lookup table** that maps each nationality (e.g., "British", "Mexican", "Japanese") to its geographic region (e.g., "Europe", "North America", "Asia"). This table is stored in the gold layer and is used by other gold-layer dimension notebooks (like `dim_constructors` and `dim_drivers`) to enrich data with regional information.

##### What this notebook does:
1. **Loads** environment configuration variables (`catalog_name`, `gold_schema`)
2. **Defines** the target table path: `formula1.gold.ref_nationalaty_regions`
3. **Creates** a Spark DataFrame manually using `Row` objects - containing all 41 unique nationalities found across the `silver.drivers` and `silver.constructors` tables
4. **Writes** the DataFrame as a Delta table to the gold schema
5. **Verifies** the written data by reading it back

##### Purpose:
This is a **static reference table** - it doesn't read from any upstream source table. Instead, the nationality-to-region mapping is manually defined based on all distinct nationality values found in the Formula 1 dataset. It enables regional analysis (e.g., "How many European constructors vs Asian ones?") across multiple dimension tables.

##### Dependencies:
- **Upstream config:** **`00-common/01.environment-config`** (provides `catalog_name`, `gold_schema`)
- **Source:** None (manually defined mappings)
- **Target table:** `formula1.gold.ref_nationalaty_regions`
- **Used by:** **`02) Build Constructors Dimention`**, **`03) Build Drivers Dimention`**

##### Step 0: Load Environment Config & Import Functions
We run the shared environment configuration notebook which defines key variables:
- `catalog_name` -> `formula1`
- `gold_schema` -> `gold`
- `silver_schema` -> `silver`

Then we import all built-in functions from `pyspark.sql.functions` for any transformations we may need.

In [0]:
%run ../00-common/01.environment-config 


In [0]:
from pyspark.sql.functions import *

##### Step 1: Define the Target Table
We define the fully qualified target table name where our reference table will be stored. The variable `target_table` is set to `formula1.gold.ref_nationalaty_regions` by combining:
- `catalog_name` -> `formula1`
- `gold_schema` -> `gold`
- Table name -> `ref_nationalaty_regions`

This reference table lives in the **gold layer** because it's a business-ready lookup table used by other gold-layer dimension tables.

##### Step 2: Create the Nationality-to-Region Mapping DataFrame
We manually create a Spark DataFrame called `nationality_region_map_rows` using `pyspark.sql.Row` objects. Each row contains:
- `nationality` (string) - the nationality value as it appears in `silver.drivers` and `silver.constructors`
- `region` (string) - the geographic region that nationality belongs to

**All 41 nationalities** found across both silver tables are mapped to one of 6 regions:

| Region | Count | Examples |
| --- | --- | --- |
| Europe | 22 | British, German, Italian, French, Dutch, Finnish |
| Asia | 7 | Chinese, Indian, Japanese, Thai, Malaysian |
| South America | 6 | Brazilian, Argentine, Colombian, Chilean |
| North America | 3 | American, Canadian, Mexican |
| Oceania | 2 | Australian, New Zealander |
| Africa | 2 | South African, Rhodesian |

The DataFrame is then assigned to `ref_nationality_region_df` and displayed for verification.

In [0]:
target_table = f'{catalog_name}.{gold_schema}.ref_nationalaty_regions'

In [0]:
from pyspark.sql import Row

nationality_region_map_rows = spark.createDataFrame([
    Row(nationality="American", region="North America"),
    Row(nationality="Argentine", region="South America"),
    Row(nationality="Australian", region="Oceania"),
    Row(nationality="Austrian", region="Europe"),
    Row(nationality="Belgian", region="Europe"),
    Row(nationality="Brazilian", region="South America"),
    Row(nationality="British", region="Europe"),
    Row(nationality="Canadian", region="North America"),
    Row(nationality="Chilean", region="South America"),
    Row(nationality="Chinese", region="Asia"),
    Row(nationality="Colombian", region="South America"),
    Row(nationality="Czech", region="Europe"),
    Row(nationality="Danish", region="Europe"),
    Row(nationality="Dutch", region="Europe"),
    Row(nationality="East German", region="Europe"),
    Row(nationality="Finnish", region="Europe"),
    Row(nationality="French", region="Europe"),
    Row(nationality="German", region="Europe"),
    Row(nationality="Hong Kong", region="Asia"),
    Row(nationality="Hungarian", region="Europe"),
    Row(nationality="Indian", region="Asia"),
    Row(nationality="Indonesian", region="Asia"),
    Row(nationality="Irish", region="Europe"),
    Row(nationality="Italian", region="Europe"),
    Row(nationality="Japanese", region="Asia"),
    Row(nationality="Liechtensteiner", region="Europe"),
    Row(nationality="Malaysian", region="Asia"),
    Row(nationality="Mexican", region="North America"),
    Row(nationality="Monegasque", region="Europe"),
    Row(nationality="New Zealander", region="Oceania"),
    Row(nationality="Polish", region="Europe"),
    Row(nationality="Portuguese", region="Europe"),
    Row(nationality="Rhodesian", region="Africa"),
    Row(nationality="Russian", region="Europe"),
    Row(nationality="South African", region="Africa"),
    Row(nationality="Spanish", region="Europe"),
    Row(nationality="Swedish", region="Europe"),
    Row(nationality="Swiss", region="Europe"),
    Row(nationality="Thai", region="Asia"),
    Row(nationality="Uruguayan", region="South America"),
    Row(nationality="Venezuelan", region="South America")
])
ref_nationality_region_df = nationality_region_map_rows

display(ref_nationality_region_df)

##### Step 3: Write to Gold Layer as Delta Table
We write the `ref_nationality_region_df` DataFrame to the gold schema as a Delta table. This makes the mapping permanently available for other notebooks to read.

- **Format:** Delta (default in Databricks - supports ACID transactions, time travel, and schema evolution)
- **Mode:** Overwrite - replaces the entire table with the latest mapping each time this notebook runs
- **Method:** `.saveAsTable(target_table)` - creates a managed Unity Catalog table at `formula1.gold.ref_nationalaty_regions`

Once written, any notebook can read this table using:
```python
spark.table('formula1.gold.ref_nationalaty_regions')
```

In [0]:
(
    ref_nationality_region_df.write
    .mode("overwrite")
    .saveAsTable(target_table)
)

##### Step 4: Verify the Written Table
As a final check, we read the table back from the gold schema using `spark.table(target_table)` and display it. This confirms that:
- The table was created successfully in Unity Catalog
- All 41 rows (nationality -> region mappings) were written correctly
- The schema matches expectations: two string columns (`nationality`, `region`)

In [0]:
display(spark.table(target_table))